# owl2 two-node hardware test on FABRIC

This notebook settles the open question from Phase 1 of `.claude/owl2-plan.md` before Phase 2 starts:

> **Can `--clock hw` be trusted on FABRIC VMs, or must the default be `--clock sw`?**

FABRIC VMs keep `CLOCK_REALTIME` PTP-synchronized via `ptp_kvm` + `phc2sys`. Hardware timestamps,
however, come from the dataplane NIC's own PTP hardware clock (PHC). If nothing disciplines that PHC,
hardware one-way latencies between two sites are meaningless, no matter how precise each stamp is.

The notebook:

1. creates a slice with **two VMs at two PTP-capable sites**, each with a FABNetv4 and a FABNetv6 NIC;
2. installs PTP (the `fabric-testbed/ptp` playbook) and owl (branch `owl2`, no pip needed);
3. **checks the clocks directly**: system clock vs `ptp_kvm`, and system clock vs each NIC's PHC,
   measured twice to get drift;
4. runs `owl ping` in **both directions × IPv4/IPv6 × hw/sw** and compares the two one-way
   latencies with the round-trip time;
5. prints a verdict and saves everything to a JSON file you can paste or attach.

Running it end to end takes about 20–30 minutes, most of it slice creation and the PTP playbook.
Re-running is safe: an existing slice with the same name is reused, and setup steps are idempotent.

In [ ]:
import datetime
import json
import shlex
import textwrap
from concurrent.futures import as_completed

import pandas as pd
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()
fablib.show_config();

## Parameters

Leave `SITES` as `[None, None]` to pick two random PTP-capable sites, or name them, e.g. `["STAR", "UTAH"]`.
Sites far apart make the test more informative: clock errors show up as asymmetry between the two directions.

In [ ]:
SLICE_NAME = "owl2-two-node-test"
SITES = [None, None]              # e.g. ["STAR", "UTAH"]; None = random PTP-capable site
AVOID = []                         # sites never to pick at random
IMAGE = "default_ubuntu_22"        # Python 3.10; the ptp playbook supports Ubuntu 20/22/24
NODE_NAMES = ["owl-a", "owl-b"]

OWL_REPO = "https://github.com/fabric-testbed/owl.git"
OWL_BRANCH = "owl2"
OWL_DIR = "~/owl"                  # owl runs as `python3 -m owl` from this checkout

PROBES = 50                        # probes per run
INTERVAL = 0.2                     # seconds between probes

## Pick two PTP-capable sites

In [ ]:
resources = fablib.get_resources()
chosen = [s for s in SITES if s]
for s in chosen:
    if not resources.get_ptp_capable(site_name=s):
        raise ValueError(f"{s} is not PTP-capable")
missing = len(SITES) - len(chosen)
if missing:
    chosen += fablib.get_random_sites(
        count=missing,
        avoid=AVOID + chosen,
        filter_function=lambda site: site.get("ptp_capable") is True,
    )
SITES = chosen
print("Sites:", dict(zip(NODE_NAMES, SITES)))

## Create (or reuse) the slice

Each VM gets two `NIC_Basic` (ConnectX SR-IOV VF) interfaces, one on FABNetv4 and one on FABNetv6.
`add_fabnet()` also adds the route to the whole FABNet range, so the VMs reach each other across sites.

In [ ]:
try:
    slice = fablib.get_slice(name=SLICE_NAME)
    print(f"Reusing existing slice {SLICE_NAME} (state {slice.get_state()})")
    SITES = [slice.get_node(n).get_site() for n in NODE_NAMES]
except Exception:
    slice = fablib.new_slice(name=SLICE_NAME)
    for name, site in zip(NODE_NAMES, SITES):
        node = slice.add_node(name=name, site=site, image=IMAGE, cores=2, ram=8, disk=20)
        node.add_fabnet(net_type="IPv4")
        node.add_fabnet(net_type="IPv6")
    slice.submit()
    slice = fablib.get_slice(name=SLICE_NAME)

nodes = {n: slice.get_node(n) for n in NODE_NAMES}
print("Lease end:", slice.get_lease_end())

### Interfaces and addresses

In [ ]:
info = {}
for name, node in nodes.items():
    site = node.get_site()
    i4 = node.get_interface(network_name=f"FABNET_IPv4_{site}")
    i6 = node.get_interface(network_name=f"FABNET_IPv6_{site}")
    info[name] = {
        "site": site,
        "dev4": i4.get_device_name(), "ip4": str(i4.get_ip_addr()),
        "dev6": i6.get_device_name(), "ip6": str(i6.get_ip_addr()),
        "mgmt": node.get_management_ip(),
    }
pd.DataFrame(info).T

In [ ]:
def run_on_all(cmd_for, quiet=True):
    """Run cmd_for(name) on every node in parallel; return {name: (stdout, stderr)}."""
    threads = {nodes[n].execute_thread(cmd_for(n)): n for n in nodes}
    out = {}
    for t in as_completed(threads):
        name = threads[t]
        try:
            out[name] = t.result()
        except Exception as e:
            out[name] = ("", f"{type(e).__name__}: {e}")
    return {n: out[n] for n in nodes}


def tail(text, lines=15):
    return "\n".join((text or "").splitlines()[-lines:])

## Install PTP and owl

Same sequence as FABlib's Crinkle (`fablib/crease/crinkle.py`), minus the kernel upgrade: stop
unattended upgrades (a new kernel without `linux-modules-extra` would lose `ptp_kvm`), install
`linux-modules-extra` for the running kernel, run the `fabric-testbed/ptp` playbook, then clone owl.
owl needs only Python ≥ 3.9 and the standard library, so there is nothing to `pip install`.

This takes 5–15 minutes.

In [ ]:
APT = ("sudo DEBIAN_FRONTEND=noninteractive apt-get -o DPkg::Lock::Timeout=600"
       " -o Dpkg::Options::=--force-confold")

SETUP_CMD = textwrap.dedent(f"""
    sudo systemctl disable --now apt-daily.timer apt-daily-upgrade.timer unattended-upgrades.service >/dev/null 2>&1
    sudo systemctl stop apt-daily.service apt-daily-upgrade.service >/dev/null 2>&1
    for _ in $(seq 300); do pgrep -f '[u]nattended-upgrade( |$)' >/dev/null || pgrep -f '[a]pt.systemd.daily' >/dev/null || break; sleep 2; done
    sudo dpkg --configure -a >/dev/null 2>&1
    {APT} update -q >/dev/null && {APT} install -y -q ansible git ethtool "linux-modules-extra-$(uname -r)" >/dev/null || echo SETUP_WARN apt
    # The playbook builds linuxptp 4.4 itself, and installs gcc/make only if no ptp4l is present.
    # Ubuntu's linuxptp package would make it skip that and fail, so remove it if an earlier run added it.
    dpkg -s linuxptp >/dev/null 2>&1 && {APT} purge -y -q linuxptp >/dev/null
    sudo modprobe ptp_kvm || echo SETUP_WARN ptp_kvm
    mkdir -p ~/.owltest && cd ~/.owltest && (test -d ptp || git clone -q https://github.com/fabric-testbed/ptp.git)
    cd ~/.owltest/ptp/ansible && ansible-playbook --connection=local --inventory 127.0.0.1, --limit 127.0.0.1 \\
        playbook_fabric_experiment_ptp.yml > ~/.owltest/ptp-playbook.log 2>&1 || echo SETUP_WARN ptp-playbook
    grep -E '^(fatal|127\\.0\\.0\\.1 +:)' ~/.owltest/ptp-playbook.log | tail -n 5
    echo "PTP_VERSION=$(/usr/sbin/ptp4l -v 2>&1)"
    sleep 5; systemctl list-units --all --plain --no-legend 'phc2sys@*' | awk '{{print "UNIT", $1, $3, $4}}'
    systemctl is-active -q phc2sys@CLOCK_REALTIME || echo SETUP_WARN phc2sys@CLOCK_REALTIME not running
    if [ -d {OWL_DIR}/.git ]; then git -C {OWL_DIR} fetch -q origin {OWL_BRANCH} && git -C {OWL_DIR} checkout -q -B {OWL_BRANCH} FETCH_HEAD;
    else git clone -q -b {OWL_BRANCH} {OWL_REPO} {OWL_DIR}; fi
    cd {OWL_DIR} && echo "OWL_COMMIT=$(git rev-parse --short HEAD)" && python3 -m owl --version
    echo SETUP_DONE
""")

setup = run_on_all(lambda n: SETUP_CMD)
for name, (out, err) in setup.items():
    ok = "SETUP_DONE" in out and "SETUP_WARN" not in out
    print(f"=== {name} ({info[name]['site']}): {'OK' if ok else 'CHECK OUTPUT'}")
    print(tail(out, 14 if ok else 30))
    if not ok and err:
        print("stderr:", tail(err))

## Clock check: is the NIC's hardware clock synchronized?

For each VM this reports:

- **`phc2sys`** — whether `phc2sys@CLOCK_REALTIME` is active (it steers the system clock from `ptp_kvm`);
- **`ptp_kvm`** — system clock minus the host's PTP clock as seen through `/dev/ptp_kvm` (should be within ~1 µs);
- **NIC PHC** — system clock minus each dataplane NIC's PHC (from `ethtool -T`), measured twice 5 s apart.

How to read the NIC PHC numbers:

| offset (system − PHC) | drift | meaning |
|---|---|---|
| ≈ −37 s, stable to µs | ≈ 0 ppm | PHC synchronized, running TAI → `--clock hw` usable |
| ≈ 0, stable to µs | ≈ 0 ppm | PHC synchronized, running UTC → `--clock hw` usable |
| anything else, or drifting | ≠ 0 | PHC free-running → `--clock hw` latencies across sites are wrong |

Each offset is the tightest of 300 system/PHC/system `clock_gettime` triples, the same method Crinkle uses.

The PTP playbook starts `phc2sys@CLOCK_REALTIME` and also `phc2sys@<iface>` for every interface with a PHC,
which tries to steer the NIC's clock from `ptp_kvm`. The cell lists those units and the tail of
`/var/log/phc2sys.log`: whether a VF is allowed to adjust its NIC's clock decides whether `--clock hw` can work.

In [ ]:
PHC_PROBE_PY = r"""
import json, os, sys, time
def offset(dev, n=300):
    fd = os.open(dev, os.O_RDONLY)
    try:
        clk = ((~fd) << 3) | 3          # FD_TO_CLOCKID
        g = time.clock_gettime_ns
        best = min((b - a, (a + b) // 2 - p) for a, p, b in
                   ((g(0), g(clk), g(0)) for _ in range(n)))
        return best[1]
    finally:
        os.close(fd)
devs = sys.argv[1:]
first = {d: offset(d) for d in devs if os.path.exists(d)}
time.sleep(5)
out = {}
for d, o1 in first.items():
    o2 = offset(d)
    out[d] = {"offset_ns": o2, "drift_ppm": (o2 - o1) / 5e3}
print("PHC_JSON=" + json.dumps(out))
"""


def clock_cmd(name):
    devs = " ".join(sorted({info[name]["dev4"], info[name]["dev6"]}))
    shell = textwrap.dedent(f"""\
        echo KERNEL=$(uname -r)
        echo PHC2SYS=$(systemctl is-active phc2sys@CLOCK_REALTIME)
        systemctl list-units --all --plain --no-legend 'phc2sys@*' | awk '{{print "UNIT=" $1 " " $3 "/" $4}}'
        echo "PHC2SYS_LOG<<EOF"; sudo tail -n 12 /var/log/phc2sys.log 2>&1; echo EOF
        phcs=/dev/ptp_kvm
        for d in {devs}; do
            ts=$(ethtool -T $d 2>/dev/null || ethtool -T ${{d%%.*}} 2>/dev/null)
            # "PTP Hardware Clock: N" (older ethtool) or "Hardware timestamp provider index: N" (newer)
            idx=$(echo "$ts" | awk -F': ' '/PTP Hardware Clock|provider index/{{print $2; exit}}')
            echo "ETHTOOL_$d<<EOF"; echo "$ts"; echo EOF
            echo "DRIVER_$d=$(ethtool -i ${{d%%.*}} 2>/dev/null | awk '/^driver/{{print $2}}')"
            echo "PHCIDX_$d=$idx"
            case "$idx" in ''|none) ;; *) phcs="$phcs /dev/ptp$idx";; esac
        done
        """)
    # the heredoc terminator must start at column 0
    return shell + "sudo python3 - $phcs <<'PYEOF'\n" + PHC_PROBE_PY.strip() + "\nPYEOF\n"


def parse_clock(name, out):
    res = {"node": name, "site": info[name]["site"]}
    for line in out.splitlines():
        for key in ("KERNEL", "PHC2SYS"):
            if line.startswith(key + "="):
                res[key.lower()] = line.split("=", 1)[1]
        if line.startswith("PHC_JSON="):
            res["phc"] = json.loads(line.split("=", 1)[1])
        for dev in (info[name]["dev4"], info[name]["dev6"]):
            if line.startswith(f"PHCIDX_{dev}="):
                res.setdefault("phc_index", {})[dev] = line.split("=", 1)[1]
            if line.startswith(f"DRIVER_{dev}="):
                res.setdefault("driver", {})[dev] = line.split("=", 1)[1]
    res["units"] = [l[5:] for l in out.splitlines() if l.startswith("UNIT=")]
    res["phc2sys_log"] = out.split("PHC2SYS_LOG<<EOF\n", 1)[-1].split("\nEOF", 1)[0] if "PHC2SYS_LOG" in out else ""
    res["ethtool_T"] = {}
    blocks = out.split("ETHTOOL_")[1:]
    for b in blocks:
        dev, rest = b.split("<<EOF", 1)
        res["ethtool_T"][dev] = rest.split("\nEOF", 1)[0].strip()
    return res


def classify_phc(offset_ns, drift_ppm):
    if abs(drift_ppm) > 0.1:
        return "FREE-RUNNING (drifting)"
    if abs(offset_ns + 37_000_000_000) < 100_000:
        return "synced, TAI"
    if abs(offset_ns) < 100_000:
        return "synced, UTC"
    return f"stable but offset {offset_ns / 1e9:+.6f} s (not synced to this VM's clock)"


clock_raw = run_on_all(clock_cmd)
clock = {n: parse_clock(n, out) for n, (out, err) in clock_raw.items()}

rows = []
for n, c in clock.items():
    print(f"=== {n} ({c['site']}) phc2sys units: {', '.join(c['units']) or 'none'}")
    print(textwrap.indent(c["phc2sys_log"] or "(no /var/log/phc2sys.log)", "    "))
    for dev, idx in c.get("phc_index", {}).items():
        print(f"--- {n} {dev} (driver {c.get('driver', {}).get(dev)}, PHC index {idx or 'none'}): ethtool -T")
        print(c["ethtool_T"].get(dev, "(no output)"))
    for dev_path, v in c.get("phc", {}).items():
        label = "ptp_kvm" if dev_path.endswith("ptp_kvm") else dev_path
        rows.append({
            "node": n, "site": c["site"], "phc2sys": c.get("phc2sys"), "clock": label,
            "offset_s (sys - clock)": round(v["offset_ns"] / 1e9, 9),
            "drift_ppm": round(v["drift_ppm"], 4),
            "verdict": ("synced" if abs(v["offset_ns"]) < 10_000 else "NOT synced")
                       if label == "ptp_kvm" else classify_phc(v["offset_ns"], v["drift_ppm"]),
        })
    if not c.get("phc"):
        print(f"!!! {n}: no clock data\n{tail(clock_raw[n][0], 30)}\n{tail(clock_raw[n][1])}")
clock_table = pd.DataFrame(rows)
clock_table

## Start a reflector on each VM

`owl reflect` runs in the background on both VMs, with hardware timestamping enabled on both FABNet NICs,
so each VM can be pinged over IPv4 and IPv6 with either clock. Its log is `/tmp/owl-reflect.log`.

In [ ]:
def reflect_cmd(name, hw=True):
    i = info[name]
    opts = " ".join(f"-I {d}" for d in sorted({i["dev4"], i["dev6"]})) if hw else "--no-hw"
    return (f"sudo pkill -f '^python3 -m owl reflect'; sleep 0.5; cd {OWL_DIR} && "
            f"(sudo setsid nohup python3 -m owl reflect {opts} -v > /tmp/owl-reflect.log 2>&1 < /dev/null &); "
            f"sleep 2; head -n 3 /tmp/owl-reflect.log; pgrep -f '^python3 -m owl reflect' >/dev/null && echo REFLECT_UP")

for n, (out, err) in run_on_all(reflect_cmd).items():
    if "REFLECT_UP" not in out:
        print(f"!!! {n}: reflector with hardware timestamps failed:\n{out.strip()} {err.strip()}")
        print(f"    restarting with --no-hw so the sw runs still work (hw runs will report no_rx_ts)")
        out, err = nodes[n].execute(reflect_cmd(n, hw=False), quiet=True)
    print(f"=== {n}: {'running' if 'REFLECT_UP' in out else 'NOT RUNNING'}")
    print(out.strip(), err.strip())

## Run `owl ping`: both directions × IPv4/IPv6 × hw/sw

Every run sends `PROBES` probes. `--json` output is parsed into the table below.
`--clock hw` runs with `sudo` because owl has to enable NIC timestamping.

In [ ]:
def ping(src, dst, af, clk):
    dst_ip = info[dst][f"ip{af}"]
    sudo = "sudo " if clk == "hw" else ""
    cmd = (f"cd {OWL_DIR} && {sudo}python3 -m owl ping --json -{af} --clock {clk} "
           f"-c {PROBES} -i {INTERVAL} {dst_ip}")
    # no execute(timeout=...): FABlib turns it into `sudo timeout N <cmd>`, which cannot run `cd`.
    # owl ping stops by itself after -c probes.
    out, err = nodes[src].execute(cmd, quiet=True)
    start, summary, probes = {}, {}, []
    for line in out.splitlines():
        if not line.startswith("{"):
            continue
        obj = json.loads(line)
        if obj.get("type") == "start":
            start = obj
        elif obj.get("type") == "summary":
            summary = obj
        elif obj.get("type") == "probe":
            probes.append(obj)
    statuses = {}
    for p in probes:
        statuses[p["status"]] = statuses.get(p["status"], 0) + 1
    row = {"src": src, "dst": dst, "af": f"IPv{af}", "clock": clk,
           "iface": start.get("iface"), "sent": summary.get("sent"), "ok": summary.get("ok"),
           "statuses": statuses, "error": tail(err, 3).strip()}
    for k in ("owl", "rtt"):
        s = summary.get(k) or {}
        for stat in ("min", "avg", "max", "mdev"):
            v = s.get(f"{stat}_ns")
            row[f"{k}_{stat}_ms"] = None if v is None else v / 1e6
    return row, probes


runs, raw_probes = [], {}
for src, dst in [(NODE_NAMES[0], NODE_NAMES[1]), (NODE_NAMES[1], NODE_NAMES[0])]:
    for af in (4, 6):
        for clk in ("hw", "sw"):
            row, probes = ping(src, dst, af, clk)
            runs.append(row)
            raw_probes[f"{src}->{dst} IPv{af} {clk}"] = probes
            print(f"{src}->{dst} IPv{af} {clk}: ok {row['ok']}/{row['sent']} {row['statuses']}"
                  f" owl avg {row['owl_avg_ms']} ms  rtt avg {row['rtt_avg_ms']} ms {row['error']}")

runs_table = pd.DataFrame(runs)
cols = ["src", "dst", "af", "clock", "iface", "ok", "sent", "owl_min_ms", "owl_avg_ms",
        "owl_mdev_ms", "rtt_min_ms", "rtt_avg_ms", "statuses", "error"]
runs_table[cols]

## Verdict

For each address family and clock, the two directions are combined:

- **sum** = min owl(A→B) + min owl(B→A). Clock offset cancels in the sum, so it should be just under
  **min rtt**; the gap is the reflector's turnaround time (typically tens to a few hundred µs).
- **apparent offset** = (min owl(A→B) − min owl(B→A)) / 2. This is the clock offset between the two
  VMs plus half the path asymmetry. With working PTP it should be small (µs, or at most a fraction of a ms
  if the paths differ). Seconds, or anything near ±37 s, means the clocks are in different timebases.
- **owl vs rtt/2** for each direction: on a symmetric path these are close.

In [ ]:
def pick(src, dst, af, clk):
    for r in runs:
        if (r["src"], r["dst"], r["af"], r["clock"]) == (src, dst, af, clk):
            return r


A, B = NODE_NAMES
verdict_rows = []
for af in ("IPv4", "IPv6"):
    for clk in ("hw", "sw"):
        ab, ba = pick(A, B, af, clk), pick(B, A, af, clk)
        row = {"af": af, "clock": clk}
        if not (ab and ba and ab["ok"] and ba["ok"]):
            row["verdict"] = "NO DATA — see statuses/errors above"
            verdict_rows.append(row)
            continue
        rtt_min = min(ab["rtt_min_ms"], ba["rtt_min_ms"])
        total = ab["owl_min_ms"] + ba["owl_min_ms"]
        offset = (ab["owl_min_ms"] - ba["owl_min_ms"]) / 2
        gap = rtt_min - total
        row.update({
            f"owl {A}->{B} min ms": round(ab["owl_min_ms"], 4),
            f"owl {B}->{A} min ms": round(ba["owl_min_ms"], 4),
            "rtt min ms": round(rtt_min, 4),
            "sum ms": round(total, 4),
            "rtt - sum (turnaround) ms": round(gap, 4),
            "apparent offset ms": round(offset, 4),
        })
        problems = []
        if min(ab["owl_min_ms"], ba["owl_min_ms"]) < 0:
            problems.append("negative owl")
        if not (-0.05 <= gap <= 5):
            problems.append("sum inconsistent with rtt")
        if abs(offset) > 0.5 * rtt_min:
            problems.append("offset larger than path asymmetry can explain")
        if abs(abs(offset) - 37_000) < 1:
            problems.append("≈37 s: TAI vs UTC mix")
        row["verdict"] = "PLAUSIBLE" if not problems else "BROKEN: " + "; ".join(problems)
        verdict_rows.append(row)

verdict_table = pd.DataFrame(verdict_rows)
verdict_table

## Save and share the results

Writes everything (slice info, clock check, per-run summaries, verdicts, every probe) to a JSON file next to
this notebook, and prints a short text summary to paste back into the owl2 discussion.

In [ ]:
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
fname = f"owl2_two_node_{SITES[0]}_{SITES[1]}_{stamp}.json"
with open(fname, "w") as f:
    json.dump({
        "created_utc": stamp, "slice": SLICE_NAME, "image": IMAGE, "owl_branch": OWL_BRANCH,
        "nodes": info, "clock": clock, "runs": runs, "verdicts": verdict_rows, "probes": raw_probes,
    }, f, indent=1, default=str)
print("Saved", fname, "\n")

commit = {n: next((l.split("=", 1)[1] for l in setup[n][0].splitlines() if l.startswith("OWL_COMMIT=")), "?")
          for n in nodes}
print(f"owl2 two-node test {stamp}: {A}@{info[A]['site']} <-> {B}@{info[B]['site']}, "
      f"image {IMAGE}, owl {commit}")
print("\nClock check:")
print(clock_table.to_string(index=False))
print("\nPer-direction runs:")
print(runs_table[["src", "dst", "af", "clock", "ok", "sent", "owl_min_ms", "owl_avg_ms",
                  "rtt_min_ms", "statuses"]].to_string(index=False))
print("\nVerdict:")
print(verdict_table.to_string(index=False))

## Optional: look at the reflector logs / interactive checks

Use these to dig into anything odd, e.g. run a longer `owl ping` by hand to watch for jumps.

In [ ]:
for n, node in nodes.items():
    print(f"=== {n}")
    node.execute("tail -n 5 /tmp/owl-reflect.log", quiet=False)

# Example: a longer hand-run ping, human-readable output
# nodes[A].execute(f"cd {OWL_DIR} && sudo python3 -m owl ping -c 20 {info[B]['ip4']}")

## Clean up

In [ ]:
# Stop the reflectors
run_on_all(lambda n: "sudo pkill -f '^python3 -m owl reflect'; echo stopped")

DELETE_SLICE = False  # set True when you are done with the VMs
if DELETE_SLICE:
    slice.delete()
    print("Slice deleted")